<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-07-npv-irr-wacc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Complemento Sesión 07 — NPV, IRR y WACC

**Módulo:** Sistemas de Big Data (5074) · **Uso:** explicación en aula (profesor) o repaso tras la [Sesión 07](https://juanarrow.github.io/BigData/sesiones/sesion-07.html).

En la sesión 07 ya calculamos el **ROI** simple. Aquí damos un paso más: **cuándo** entra el dinero importa. Solo usa la **biblioteca estándar de Python** (sin `pip install`).


## 1. Recordatorio: ROI vs valor en el tiempo

| | ROI (sesión 07) | NPV / IRR |
|---|---|---|
| Pregunta | ¿Cuánto gano respecto al coste total? | ¿Cuánto vale **hoy** el proyecto? ¿Supera el mínimo de la empresa? |
| Calendario | Suele mezclar varios años | Cada año (o mes) cuenta por separado |

**Objetivos de este cuaderno:**
- Calcular un **WACC** sencillo (o usar uno dado).
- Calcular **NPV** paso a paso y con función reutilizable.
- Obtener la **IRR** y compararla con el WACC.
- Aplicar el **caso de logística** del artículo DATAVERSITY (lectura complementaria de la sesión 07).


## 2. WACC — coste medio ponderado del capital

Es la **rentabilidad mínima** que la empresa exige a un proyecto para compensar cómo se financia (capital + deuda).

$$\mathrm{WACC} = \frac{E}{V} \cdot R_e + \frac{D}{V} \cdot R_d \cdot (1 - T_c)$$

- $E$ = valor del capital (equity), $D$ = deuda, $V = E + D$
- $R_e$ = rentabilidad exigida por accionistas, $R_d$ = tipo de interés de la deuda
- $T_c$ = tipo impositivo (el impuesto reduce el coste efectivo de la deuda)

En muchos ejercicios el WACC **ya viene dado** (p. ej. 10 %). Lo usamos como **tasa de descuento** del NPV y como **nota de corte** frente a la IRR.


In [ ]:
def calcular_wacc(equity, deuda, re, rd, tipo_impositivo):
    """WACC en tanto por uno (0.10 = 10 %)."""
    v = equity + deuda
    if v <= 0:
        raise ValueError('E + D debe ser positivo')
    return (equity / v) * re + (deuda / v) * rd * (1 - tipo_impositivo)

# Ejemplo numérico (millones de €)
E, D = 60, 40
Re, Rd, Tc = 0.12, 0.05, 0.25
wacc = calcular_wacc(E, D, Re, Rd, Tc)
print(f'WACC = {wacc * 100:.2f} %')
print('(En el caso DATAVERSITY usaremos WACC = 10 % directamente)')


## 3. NPV — valor actual neto

Traemos todos los flujos de caja a **valor de hoy** dividiendo entre $(1 + r)^t$, con $r$ = tasa de descuento (normalmente el WACC).

$$\mathrm{NPV} = \sum_{t=0}^{n} \frac{CF_t}{(1 + r)^t}$$

Convención: $CF_0$ suele ser **negativo** (inversión inicial). Si **NPV > 0**, el proyecto **crea valor** con esa tasa de descuento.


In [ ]:
def npv(tasa, flujos):
    """
    flujos: lista [CF0, CF1, ..., CFn] en orden temporal.
    tasa: decimal anual (0.10 = 10 %).
    """
    return sum(cf / (1 + tasa) ** t for t, cf in enumerate(flujos))

def tabla_npv(tasa, flujos):
    """Muestra desglose año a año (para explicar en clase)."""
    filas = []
    for t, cf in enumerate(flujos):
        factor = (1 + tasa) ** t
        presente = cf / factor
        filas.append((t, cf, factor, presente))
    return filas

# Mini-ejemplo: inviertes 1000 € y recibes 400 €/año durante 3 años
flujos_demo = [-1000, 400, 400, 400]
r = 0.10
for t, cf, factor, pv in tabla_npv(r, flujos_demo):
    print(f'Año {t}: flujo {cf:>8.0f} €  ÷ {factor:>6.3f}  =  {pv:>8.2f} € hoy')
print(f'NPV total = {npv(r, flujos_demo):.2f} €')


## 4. IRR — tasa interna de retorno

Es la tasa $r$ que hace **NPV = 0**. No se despeja fácil a mano; usamos un método numérico (búsqueda binaria en este cuaderno).

**Regla práctica (como en DATAVERSITY):**
- Si **NPV > 0** y **IRR > WACC** → el proyecto suele **merecer la pena**.
- Si no → conviene replantear o comparar con otras inversiones.


In [ ]:
def irr(flujos, tasa_min=-0.99, tasa_max=5.0, pasos=200):
    """IRR por búsqueda binaria (NPV = 0). tasa en tanto por uno."""
    lo, hi = tasa_min, tasa_max
    nplo, nphi = npv(lo, flujos), npv(hi, flujos)
    if nplo * nphi > 0:
        raise ValueError('No se encontró IRR en el intervalo; prueba otro rango.')
    for _ in range(pasos):
        mid = (lo + hi) / 2
        if abs(npv(mid, flujos)) < 1e-9:
            return mid
        if npv(lo, flujos) * npv(mid, flujos) <= 0:
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2

tasa_irr = irr(flujos_demo)
print(f'IRR del mini-ejemplo = {tasa_irr * 100:.2f} %')
print(f'NPV con esa IRR (debe ≈ 0): {npv(tasa_irr, flujos_demo):.6f}')


## 5. Caso completo: proyecto de analítica (DATAVERSITY)

Empresa de logística: inversión inicial **235 000 €** y beneficios anuales durante 5 años. WACC = **10 %**.

Beneficios (€/año): 54 000 · 70 000 · 83 000 · 96 000 · 106 000


In [ ]:
WACC = 0.10
flujos_logistica = [-235_000, 54_000, 70_000, 83_000, 96_000, 106_000]

print('=== Desglose NPV (WACC 10 %) ===')
for t, cf, factor, pv in tabla_npv(WACC, flujos_logistica):
    print(f'Año {t}: {cf:>10,.0f} €  →  valor hoy {pv:>10,.2f} €')

valor_npv = npv(WACC, flujos_logistica)
tasa_irr_log = irr(flujos_logistica)

print()
print(f'NPV  = {valor_npv:,.2f} €')
print(f'IRR  = {tasa_irr_log * 100:.2f} %')
print(f'WACC = {WACC * 100:.0f} %')
print()
if valor_npv > 0 and tasa_irr_log > WACC:
    print('✅ NPV > 0 e IRR > WACC → encaja con invertir (según el artículo).')
else:
    print('❌ No cumple la regla NPV/IRR vs WACC.')


## 6. Comparar con un ROI “simple” (misma historia, otra mirada)

El ROI de la sesión 07 **no descuenta** el calendario. Aquí sumamos todos los beneficios, restamos la inversión inicial **sin** descontar años, y dividimos entre un coste total **aproximado**.

> Ojo: no es el mismo ROI que el notebook de la sesión 07 (allí entran costes anuales de operación). Sirve para **contrastar** métodos en clase.


In [ ]:
inversion = 235_000
beneficios_anuales = flujos_logistica[1:]
beneficio_bruto = sum(beneficios_anuales)
beneficio_neto_simple = beneficio_bruto - inversion
roi_simple = beneficio_neto_simple / inversion * 100

print(f'Suma beneficios 5 años: {beneficio_bruto:,.0f} €')
print(f'Beneficio neto (sin descontar): {beneficio_neto_simple:,.0f} €')
print(f'ROI simple = {roi_simple:.1f} %')
print(f'NPV (con WACC 10 %): {valor_npv:,.0f} €  → mira el calendario, no solo el total')


## 7. Prueba tú (en clase o en casa)

1. Cambia el **WACC** a 15 % y vuelve a calcular el NPV del caso logística. ¿Sigue siendo positivo?
2. Inventa flujos `[-50_000, 20_000, 20_000, 20_000]` y calcula NPV con WACC 10 % e IRR.
3. Relaciona con la pregunta de la sesión 07: *¿Cómo justificarías la inversión en un proyecto Big Data?* (menciona costes, beneficios **atribuibles** y, si el proyecto es largo, NPV/IRR).


In [ ]:
# Espacio para experimentar
WACC_alumno = 0.15  # prueba cambiando este valor
print(f'NPV con WACC {WACC_alumno*100:.0f} %:', f'{npv(WACC_alumno, flujos_logistica):,.2f} €')


---

**Referencias:** [Sesión 07 — ROI y calidad](https://juanarrow.github.io/BigData/sesiones/sesion-07.html) · [DATAVERSITY — NPV, IRR, WACC](https://www.dataversity.net/articles/a-financial-approach-to-evaluating-data-analytics-and-ai-investments/)
